In [40]:
import pandas as pd
import numpy as np
import xailib.xailib_base

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer
from lore_explainer.explanation import ExplanationEncoder
from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt

import os

In [7]:
from plot_explanation import PlotExplanation
import pickle

In [8]:
path = os.getcwd()
print(path)

/home/sax/PycharmProjects/xai-visualization_rules_fi/notebooks


# Loading a dataset and preparing it

In [9]:
datasets=['titanic_c.csv','german_credit.csv']

In [10]:
source_file = f'../datasets/{datasets[1]}'
class_field = 'default'
# Load and transform dataset
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [11]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

## Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [12]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [13]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [14]:
inst_num = 34
inst = X_train.iloc[inst_num].values
print('Instance ',inst)
print('True class ',Y_train.iloc[inst_num])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [  18 6458    2    4   39    2    2    0    0    0    1    1    0    0
    0    0    0    0    1    0    0    0    0    0    0    0    0    1
    0    0    0    1    0    0    0    0    0    0    0    1    0    0
    1    0    0    0    1    1    0    0    0    1    0    1    0    0
    0    0    1    0    1]
True class  1
Predicted class  [1]


## SHAP Explainer

In [15]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer': 'tree', 'X_train': X_train.iloc[0:].values}
explainer.fit(config)
exp = explainer.explain(inst)
shap_feature_importance = exp.exp

In [33]:
len(shap_feature_importance[0])

61

### LORE explainer

In [16]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':10000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)

exp = explainer.explain(inst)
print(exp)

In [17]:
expDict = exp.expDict
# remove key dt from expDict
expDict.pop('dt', None)
expDict

{'bb_pred': 1,
 'dt_pred': 1,
 'rule': {'premise': [{'att': 'other_installment_plans=bank',
    'op': '>',
    'thr': 0.9083859324455261,
    'is_continuous': True},
   {'att': 'credit_amount', 'op': '>', 'thr': 4226.0, 'is_continuous': True},
   {'att': 'duration_in_month', 'op': '>', 'thr': 15.5, 'is_continuous': True},
   {'att': 'savings=... < 100 DM',
    'op': '>',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'age', 'op': '<=', 'thr': 41.0, 'is_continuous': True},
   {'att': 'purpose=car (used)',
    'op': '<=',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'telephone=none', 'op': '<=', 'thr': 0.5, 'is_continuous': True},
   {'att': 'credits_this_bank', 'op': '>', 'thr': 0.5, 'is_continuous': True},
   {'att': 'account_check_status=< 0 DM',
    'op': '<=',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'credit_history=existing credits paid back duly till now',
    'op': '<=',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'property=real estate',

In [18]:
rules =exp.expDict['rule']
crules = exp.expDict['crules']

In [19]:
rules

{'premise': [{'att': 'other_installment_plans=bank',
   'op': '>',
   'thr': 0.9083859324455261,
   'is_continuous': True},
  {'att': 'credit_amount', 'op': '>', 'thr': 4226.0, 'is_continuous': True},
  {'att': 'duration_in_month', 'op': '>', 'thr': 15.5, 'is_continuous': True},
  {'att': 'savings=... < 100 DM',
   'op': '>',
   'thr': 0.5,
   'is_continuous': True},
  {'att': 'age', 'op': '<=', 'thr': 41.0, 'is_continuous': True},
  {'att': 'purpose=car (used)', 'op': '<=', 'thr': 0.5, 'is_continuous': True},
  {'att': 'telephone=none', 'op': '<=', 'thr': 0.5, 'is_continuous': True},
  {'att': 'credits_this_bank', 'op': '>', 'thr': 0.5, 'is_continuous': True},
  {'att': 'account_check_status=< 0 DM',
   'op': '<=',
   'thr': 0.5,
   'is_continuous': True},
  {'att': 'credit_history=existing credits paid back duly till now',
   'op': '<=',
   'thr': 0.5,
   'is_continuous': True},
  {'att': 'property=real estate',
   'op': '<=',
   'thr': 0.5,
   'is_continuous': True},
  {'att': 'pres

In [20]:
exp.expDict['crules']

[{'premise': [{'att': 'other_installment_plans=bank',
    'op': '>',
    'thr': 0.9083859324455261,
    'is_continuous': True},
   {'att': 'credit_amount', 'op': '>', 'thr': 4226.0, 'is_continuous': True},
   {'att': 'duration_in_month', 'op': '>', 'thr': 15.5, 'is_continuous': True},
   {'att': 'savings=... < 100 DM',
    'op': '>',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'age', 'op': '<=', 'thr': 41.0, 'is_continuous': True},
   {'att': 'purpose=car (used)',
    'op': '<=',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'telephone=none', 'op': '<=', 'thr': 0.5, 'is_continuous': True},
   {'att': 'credits_this_bank', 'op': '>', 'thr': 0.5, 'is_continuous': True},
   {'att': 'account_check_status=< 0 DM',
    'op': '>',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'job=skilled employee / official',
    'op': '<=',
    'thr': 0.5,
    'is_continuous': True}],
  'cons': 0,
  'class_name': 'default'}]

# Plotting functions

In [21]:
def single_feature_importance_plot(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).mark_bar(
    ).encode(
        x=alt.X(
            field='feature_importance',
            type='quantitative',
            title=None
        ),
        y=alt.Y(
            field='name',
            type='nominal',
            title=None,
            axis=None
        ),
        color=alt.condition('datum.feature_importance > 0', alt.value('#285588'), alt.value('#E36273')),
        tooltip=[alt.Tooltip(field="name"),alt.Tooltip(field="feature_importance")]
    )
    return chart.properties(
        height=20,
        width=100
    )

In [22]:
def single_rule_plot_numeric(dataframe,rw):
    data = dataframe[dataframe['name'] == rw['name']]
    p=alt.Chart(
        data
    ).mark_point(
        color='black' if rw['is_continuous'] == True else 'black',
        size=70,
        shape='diamond',
        filled=True
    ).encode(
        x=alt.X(
            field='inst',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        tooltip=[alt.Tooltip(field='inst', title=rw['name'])]
    )

    t_min = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=-10,
        align='right'
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None
        ),
        text='min:N'
    )

    t_max = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=10,
        align='left'
    ).encode(
        x=alt.X(
            field='max',
            type='quantitative',
            title=None
        ),
        text='max:N'
    )

    

    b =alt.Chart(
        data
    ).mark_bar(
        color='#fcc40f',size=5,
        stroke='white'
    ).encode(
        x=alt.X(
            field='thr',
            type='quantitative',
            title=None,
        ),
        x2='thr2',
        y=alt.Y(
            field='name',
            type='nominal',
            title=None
        ),

    )



    l =alt.Chart(
        data
    ).mark_bar(
        color='grey',size=1
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2='max',
        y=alt.Y(field='name',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
    )


    q1_m = alt.Chart(
        data
    ).mark_bar(
        stroke='white',
        color='lightgrey',
        size=18
    ).encode(
        x=alt.X(
            field='q1',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='median'
        ),
    )

    m_q3 = alt.Chart(
        data
    ).mark_bar(
        stroke='white',
        color='lightgrey',
        size=18,
    ).encode(
        x=alt.X(
            field='median',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='q3'
        )
    )
    
    return alt.layer(l,q1_m,m_q3,b,p).properties(
        height=20,
        width=300        
    )

In [23]:
def single_index_text(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).transform_calculate(
        label ="datum.type=='categorical' ? datum.name : datum.name +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
    ).mark_text(
        color='black',
        align='left',
        dx=-50,
        fontSize=13
    ).encode(
            text=alt.Text(
            field='label',
            type='nominal',
            title=None
        )
    )
    return chart.properties(
        height=20,
        width=101
    )

In [24]:
def plot_rules(dataframe, only_rules=False):
    ti_list=[]
    rp_list=[]
    fi_list=[]
    
    for i, row in dataframe.iterrows():
        if row['inst']!=0:
            if ((only_rules == True) and (row['is_continuous']!= True)):
                pass
            else:
                sti = single_index_text(dataframe, row)
                if row['type']== 'numeric':
                    srp = single_rule_plot_numeric(dataframe, row)
                else:
                    srp = single_rule_plot_qualit(dataframe, row)
                sfi = single_feature_importance_plot(dataframe, row)
                ti_list.append(sti)
                rp_list.append(srp)
                fi_list.append(sfi)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI').resolve_scale(
    x='shared'
)
    final_chart = alt.hconcat(
        fi_concat, rp_concat, ti_concat
    )

    return final_chart.configure(
       # background='#F5F5F5',
        padding=20
    ).configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

In [25]:
def single_rule_plot_qualit(dataframe, rw):
    name= rw['name'].split('=')[0]
    data = dataframe[dataframe['rname'] == name]
    
    base= alt.Chart(
        data
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'descending')]
    ).transform_calculate(
        midStack='(datum.count_start+datum.count_end)/2'
    )
    
    
    bar = base.mark_bar(
        stroke='white',
        color='lightgrey'
    ).encode(
        x='count_start:Q',
        x2='count_end:Q',
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    bar_r = base.mark_bar(
         stroke='#fcc40f'
     ).encode(
        x='count_start:Q',
        x2='count_end:Q',
        color=alt.condition('datum.is_continuous && datum.inst==1',alt.value("#fcc40f"),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.0001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )

    r=base.mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='name:N',
        color=alt.condition('datum.is_continuous',alt.value('#fcc40f'),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    dot =base.mark_point(
        size=70,
        shape='diamond',
        color='black',
        filled=True
    ).encode(
        x=alt.X(
            field='midStack',
            type='quantitative',
            title=None,
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
    )
    

    return alt.layer(bar,bar_r,dot).properties(
        height=20,
        width=300,
    )

## Prepare data to plot

In [26]:
pe = PlotExplanation(
    feature_names=feature_names,
    real_feature_names=real_feature_names,
    instance_number=inst_num,
    x_train=X_train,
    rules=rules, crules=crules,
    feature_importance_type='shap',
    feature_importance=shap_feature_importance,
    numeric_columns=numeric_columns
)

In [27]:
df_v = pe.prepare_dataframe()

In [28]:
df_v

,type,name,rname,min,max,q1,median,q3,mean,std,feature_importance,category,count,inst,op,thr,is_continuous,thr2
48,categorical,other_installment_plans=none,other_installment_plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.091419,none,563.0,0,NaN,NaN,NaN,NaN
11,categorical,credit_history=all credits at this bank paid b...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.080437,all credits at this bank paid back duly,38.0,1,NaN,NaN,NaN,NaN
47,categorical,other_installment_plans=bank,other_installment_plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.063384,bank,99.0,1,>,0.908386,True,NaN
46,categorical,property=unknown / no property,property,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.039427,unknown / no property,111.0,1,NaN,NaN,NaN,NaN
1,numeric,credit_amount,credit_amount,338.0,15945.0,1360.75,2319.5,3974.5,3200.872857,2674.942042,0.027655,NaN,NaN,6458,>,4226.000000,True,15945.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22,categorical,purpose=furniture/equipment,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000198,furniture/equipment,6.0,0,NaN,NaN,NaN,NaN
9,categorical,account_check_status=>= 200 DM / salary assign...,account_check_status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000178,>= 200 DM / salary assignments for at least 1 ...,44.0,0,NaN,NaN,NaN,NaN
39,categorical,personal_status_sex=male : single,personal_status_sex,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000124,male : single,386.0,1,NaN,NaN,NaN,NaN
25,categorical,purpose=retraining,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000059,retraining,5.0,0,NaN,NaN,NaN,NaN


In [34]:
rule_descr = pe.prepare_rule_descriptor()

In [41]:
import json

with open(f'instance_{inst_num},json', "w") as outfile:
    json.dump(rule_descr, outfile, cls=ExplanationEncoder, indent=4)

# Plot

In [52]:
rules_all=plot_rules(df_v, only_rules=False)
rules_all

alt.HConcatChart(...)

In [29]:
rules_only = plot_rules(df_v, only_rules=True)
rules_only

alt.HConcatChart(...)